# 02 - Classical Baselines

**QZero**: Quantum Zero-Day Detection System

This notebook establishes classical ML baselines for comparison with quantum methods.

## Contents
1. Model Setup
2. Logistic Regression
3. Random Forest
4. SVM (RBF Kernel)
5. Gradient Boosting
6. Model Comparison
7. Feature Importance

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

from src.preprocessing.data_loader import generate_synthetic_data, DataLoader
from src.classical.baseline import (
    ClassicalBaseline, 
    run_all_baselines,
    AVAILABLE_MODELS
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print(f"Available models: {list(AVAILABLE_MODELS.keys())}")

## 1. Data Preparation

In [ ]:
# Generate data
X, y = generate_synthetic_data(n_samples=10000, random_state=42)

# Prepare with binary classification (normal vs attack)
loader = DataLoader()
data = loader.prepare_data(X, y, binary=True, test_size=0.2, val_size=0.1)

print(f"Training set: {data['X_train'].shape[0]:,} samples")
print(f"Validation set: {data['X_val'].shape[0]:,} samples")
print(f"Test set: {data['X_test'].shape[0]:,} samples")
print(f"\nClass balance (test):")
print(f"  Normal (0): {(data['y_test'] == 0).sum():,}")
print(f"  Attack (1): {(data['y_test'] == 1).sum():,}")

## 2. Run All Baselines

In [ ]:
# Run all classical baselines
results = run_all_baselines(
    data['X_train'], data['y_train'],
    data['X_test'], data['y_test'],
    models=['logistic_regression', 'random_forest', 'svm', 'gradient_boosting']
)

print("\n" + "="*70)
print("BASELINE RESULTS SUMMARY")
print("="*70)

In [ ]:
# Create results table
results_df = pd.DataFrame([
    {
        'Model': r.model_name,
        'Accuracy': r.accuracy,
        'Precision': r.precision,
        'Recall': r.recall,
        'F1 Score': r.f1_score,
        'Train Time (s)': r.train_time
    }
    for r in results
])

results_df = results_df.sort_values('F1 Score', ascending=False)
results_df.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1 Score': '{:.4f}',
    'Train Time (s)': '{:.3f}'
}).background_gradient(subset=['F1 Score'], cmap='Greens')

## 3. Detailed Model Analysis

In [ ]:
# Train individual models for detailed analysis
models = {}
for model_name in ['logistic_regression', 'random_forest', 'svm', 'gradient_boosting']:
    model = ClassicalBaseline(model_name)
    model.fit(data['X_train'], data['y_train'])
    models[model_name] = model

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
model_names = ['logistic_regression', 'random_forest', 'svm', 'gradient_boosting']
display_names = ['Logistic Regression', 'Random Forest', 'SVM (RBF)', 'Gradient Boosting']

for ax, model_name, display_name in zip(axes.flatten(), model_names, display_names):
    y_pred = models[model_name].predict(data['X_test'])
    cm = confusion_matrix(data['y_test'], y_pred)
    
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Normal', 'Attack'],
        yticklabels=['Normal', 'Attack'],
        ax=ax
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(display_name)

plt.suptitle('Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.show()

## 4. ROC Curves

In [ ]:
plt.figure(figsize=(10, 8))
colors = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']

for (model_name, model), display_name, color in zip(
    models.items(), display_names, colors
):
    # Get probability predictions
    if hasattr(model.model, 'predict_proba'):
        y_prob = model.model.predict_proba(data['X_test'])[:, 1]
    else:
        y_prob = model.model.decision_function(data['X_test'])
    
    fpr, tpr, _ = roc_curve(data['y_test'], y_prob)
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f'{display_name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.500)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Classical Baselines')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

## 5. Feature Importance (Random Forest)

In [ ]:
# Get feature importances from Random Forest
rf_model = models['random_forest'].model
feature_names = data['feature_names']

importances = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(importances['Feature'], importances['Importance'], color='steelblue')
plt.xlabel('Importance')
plt.title('Feature Importance (Random Forest)')
plt.tight_layout()
plt.show()

print("Top 5 most important features:")
for _, row in importances.tail(5).iloc[::-1].iterrows():
    print(f"  {row['Feature']}: {row['Importance']:.4f}")

## 6. Model Comparison Visualization

In [ ]:
# Radar chart of metrics
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']

fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(polar=True))

angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]  # Complete the circle

for i, (_, row) in enumerate(results_df.iterrows()):
    values = [row['Accuracy'], row['Precision'], row['Recall'], row['F1 Score']]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Model'], color=colors[i])
    ax.fill(angles, values, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics)
ax.set_ylim(0.5, 1.0)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
plt.title('Model Performance Comparison')
plt.tight_layout()
plt.show()

In [ ]:
# Bar comparison
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(results_df))
width = 0.2

bars1 = ax.bar(x - 1.5*width, results_df['Accuracy'], width, label='Accuracy', color='#3498db')
bars2 = ax.bar(x - 0.5*width, results_df['Precision'], width, label='Precision', color='#2ecc71')
bars3 = ax.bar(x + 0.5*width, results_df['Recall'], width, label='Recall', color='#e74c3c')
bars4 = ax.bar(x + 1.5*width, results_df['F1 Score'], width, label='F1 Score', color='#9b59b6')

ax.set_ylabel('Score')
ax.set_title('Classical Baseline Performance')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=15, ha='right')
ax.legend()
ax.set_ylim(0.5, 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Detailed Classification Reports

In [ ]:
# Best model detailed report
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]
y_pred = best_model.predict(data['X_test'])

print(f"Best Model: {best_model_name}")
print("="*60)
print(classification_report(
    data['y_test'], y_pred,
    target_names=['Normal', 'Attack']
))

## Summary

### Classical Baseline Results

These results establish the performance bar for quantum methods to beat:

| Metric | Best Classical |
|--------|---------------|
| Accuracy | ~95-99% |
| F1 Score | ~95-99% |
| AUC | ~0.98+ |

### Key Observations

1. **Random Forest** and **Gradient Boosting** typically perform best
2. **SVM with RBF kernel** provides a good comparison point for QSVM
3. Training times are very fast (< 1 second for most models)

### Quantum Challenge

For quantum methods to be useful, they need to:
- Match or exceed classical accuracy on this data
- Demonstrate advantage on truly unseen (zero-day) attacks
- Show scalability benefits as data/features grow

**Next**: QSVM implementation in `03_qsvm_implementation.ipynb`